# Modelo: VisualCF

- Embeddings: CLIP ViT-B/32 (512 dimensiones)

In [1]:
import sys
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
sys.path.append("..")
from src.evaluation import temporal_train_test_split, evaluate_model
from src.image_features import CLIPFeatureExtractor

MODEL_NAME = "visualcf"
RESULTS_DIR = f"../results/{MODEL_NAME}"
N_FACTORS = 50


## Definición del modelo

In [2]:
class VisualCF:
    """
    Método para combinar la predicción de SVD (filtrado colaborativo) con
    la similitud coseno entre el perfil visual del usuario y los
    embeddings de los items obtenidos con CLIP.

    score(u,i) = alpha * norm(SVD_score(u,i)) + (1-alpha) * cosine(user_visual_profile, clip_emb_i)

    El perfil del usuario se calcula a partir del promedio de los embeddings
    de los items obtenidos con CLIP que son valorados con rating >= like_threshold en train.
    """

    def __init__(self, image_embeddings, svd_model, alpha=0.5, like_threshold=4.0):
        self.alpha = alpha
        self.threshold = like_threshold
        self.svd = svd_model

        # Preparación de embeddings
        self.b_ids = list(image_embeddings.keys())
        self.emb_matrix = np.array(list(image_embeddings.values()))
        self.b_idx_map = {b_id: i for i, b_id in enumerate(self.b_ids)}

    def get_user_profile(self, user_id, reviews_df):
        # Filtramos los reviews del usuario
        user_reviews = reviews_df[reviews_df["user_id"] == user_id]
        if user_reviews.empty:
            return None

        liked_df = user_reviews[user_reviews["stars"] >= self.threshold]

        # Si no hay likes por encima del threshold, usamos todos sus reviews como fallback
        if not liked_df.empty:
            target_df = liked_df
        else:
            target_df = user_reviews

        valid_bids = [b for b in target_df["business_id"] if b in self.b_idx_map]
        if not valid_bids:
            return None

        # Promediamos los embeddings de los locales visitados
        indices = [self.b_idx_map[b] for b in valid_bids]
        return np.mean(self.emb_matrix[indices], axis=0)

    def get_normalized_svd(self, user_id):
        if user_id not in self.svd["user_index"]:
            return {}

        u_idx = self.svd["user_index"][user_id]
        raw_scores = self.svd["predicted"][u_idx]

        vmin = raw_scores.min()
        vmax = raw_scores.max()
        denom = vmax - vmin
        # Evitamos división por cero
        if denom == 0.0:
            denom = 1.0

        idx_to_b = {v: k for k, v in self.svd["item_index"].items()}

        return {
            idx_to_b[j]: float((score - vmin) / denom)
            for j, score in enumerate(raw_scores)
            if j in idx_to_b
        }

    def recommend(self, user_id, train_reviews, top_k=10):
        seen = set(train_reviews[train_reviews["user_id"] == user_id]["business_id"])

        cf_scores = self.get_normalized_svd(user_id)
        profile = self.get_user_profile(user_id, train_reviews)

        img_scores = {}
        if profile is not None:
            sims = cosine_similarity(profile.reshape(1, -1), self.emb_matrix)[0]
            img_scores = dict(zip(self.b_ids, sims))

        final_scores = {}
        for b_id in self.b_ids:
            if b_id in seen:
                continue

            cf_part = self.alpha * cf_scores.get(b_id, 0.0)
            img_part = (1 - self.alpha) * img_scores.get(b_id, 0.0)
            final_scores[b_id] = cf_part + img_part

        return sorted(final_scores, key=final_scores.get, reverse=True)[:top_k]

## Datos y embeddings

In [3]:
reviews = pd.read_csv('../data/processed/reviews.csv', parse_dates=['date'])
train_reviews, test_reviews = temporal_train_test_split(reviews, test_fraction=0.2)

image_embeddings = CLIPFeatureExtractor.load('clip_embeddings.npz')
image_label = 'CLIP (512d)'

print(f'Image model: {image_label} | {len(image_embeddings)} items')

Train: 83256 reviews | Test: 17191 reviews
Image model: CLIP (512d) | 3824 items


## Entrenar SVD base

In [4]:
users = train_reviews["user_id"].unique()
items = train_reviews["business_id"].unique()
user_idx = {u: i for i, u in enumerate(users)}
item_idx = {b: i for i, b in enumerate(items)}

rows = train_reviews["user_id"].map(user_idx)
cols = train_reviews["business_id"].map(item_idx)
vals = train_reviews["stars"].astype(float)
matrix = csr_matrix((vals, (rows, cols)), shape=(len(users), len(items)))

svd = TruncatedSVD(n_components=N_FACTORS, random_state=42)
predicted = svd.fit_transform(matrix) @ svd.components_

svd_model = {"predicted": predicted, "user_index": user_idx, "item_index": item_idx}
print(f"SVD: {len(users)} users x {len(items)} items, k={N_FACTORS}")
print(f"Explained variance ratio: {svd.explained_variance_ratio_.sum():.3f}")


SVD: 10490 users x 1151 items, k=50
Explained variance ratio: 0.297


## Evaluación

In [5]:
model = VisualCF(image_embeddings, svd_model, alpha=0.5, like_threshold=4.0)
metrics = evaluate_model(
    lambda uid, top_k: model.recommend(uid, train_reviews, top_k),
    test_reviews, train_reviews, k_values=[5, 10, 20]
)
print(f'VisualCF (alpha=0.5, like_threshold=4.0):')
print(metrics.round(4))

VisualCF (alpha=0.5, like_threshold=4.0):
    precision  recall    ndcg
K                            
5      0.0244  0.0794  0.0567
10     0.0210  0.1351  0.0761
20     0.0170  0.2123  0.0979


## Guardar resultados

In [6]:
os.makedirs(RESULTS_DIR, exist_ok=True)
metrics.to_csv(f'{RESULTS_DIR}/metrics.csv')
with open(f'{RESULTS_DIR}/config.json', 'w') as f:
    json.dump({'model': 'VisualCF', 'alpha': 0.5, 'n_factors': N_FACTORS,
               'image_model': image_label, 'like_threshold': 4.0}, f, indent=2)
print(f'Saved -> results/{MODEL_NAME}/')

Saved -> results/visualcf/
